# Phase 1 Round 2 — MulT Emotion Classification (P1 Improvements)

**Ngày:** 09/06/2026  
**Mục tiêu:** Cải thiện MulT Emotion Classification trên CMU-MOSEI với các P1 improvements:

### Improvements được áp dụng:

| # | Improvement | Priority | Impact |
|:---:|:---|:---:|:---:|
| P0.1 | Per-emotion threshold tuning | Không cần train lại | +50-85% Mean F1 |
| P1.1 | BCEWithLogitsLoss với pos_weight | Cần train lại | +30-80% Mean F1 |
| P1.2 | Focal Loss thay BCE | Cần train lại | Tự động handle imbalance |
| P1.3 | MulT P1: d_model=128, num_heads=8, stochastic depth | Cần train lại | +5-8% overall |
| P1.4 | Early stopping dựa trên mean_f1 | Cần train lại | Đúng metric |

### Cách chạy:

```
1. Round 2A: Chạy P0 (threshold tuning) — nhanh nhất, không cần train
2. Round 2B: Train MulT P1 với Focal Loss — chính
```

### Benchmark kỳ vọng:

| Metric | Baseline (P0) | Target Round 2 |
|:---|:---:|:---:|
| Happy F1 | 0.4709 | 0.55-0.65 |
| Sad F1 | ~0.27 | 0.35-0.42 |
| Angry F1 | ~0.19 | 0.28-0.35 |
| Disgust F1 | ~0.17 | 0.22-0.30 |
| Surprise F1 | ~0.10 | 0.15-0.25 |
| Fear F1 | 0.0348 | 0.08-0.15 |
| **Mean F1** | **0.2064** | **0.28-0.38** |

> **Lưu ý:** MOSEI Emotion là bài toán CỰC KỲ KHÓ. Ngay cả SOTA 2025 cũng chỉ đạt Micro F1 ~0.214. Mean F1 ~0.35-0.38 là kết quả xuất sắc cho dataset này.

## 0. Setup

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

In [ ]:
import os
import sys
from pathlib import Path

RUNTIME_PROFILE = 'colab'
REPO_SOURCE = 'git'
REPO_URL = 'https://github.com/Kandesfx/Training-Multimodal-Emotion-Analysis.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/BCDA')
REPO_PATH = Path('/content/BCDA')
USE_DRIVE_OUTPUTS = True
USE_GCS = True
GCS_BUCKET = 'mer-data-bucket-kandesfx'
WANDB_ENABLE = True
WANDB_PROJECT = 'bcda-phase1-round2'

if RUNTIME_PROFILE != 'colab':
    raise ValueError('Notebook này hiện được tối ưu cho profile colab.')

if REPO_SOURCE == 'git':
    if not REPO_PATH.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_PATH}')
    else:
        print(f'Repo already exists at {REPO_PATH}')
else:
    REPO_PATH = DRIVE_ROOT

%cd {REPO_PATH}
if str(REPO_PATH) not in sys.path:
    sys.path.append(str(REPO_PATH))

# Upgrade pip và install dependencies
!python -m pip install -q --upgrade pip
!python -m pip install -q torch torchvision torchaudio numpy pandas scikit-learn matplotlib seaborn tqdm wandb

# Authenticate GCS
if IN_COLAB and USE_GCS:
    from google.colab import auth
    print('Authenticating for GCS access...')
    auth.authenticate_user()
    print('Downloading aligned_50.pkl from GCS...')
    get_ipython().system(f'mkdir -p /content/data/MSA-Dataset')
    get_ipython().system(f'gsutil cp gs://{GCS_BUCKET}/data/MSA-Dataset/aligned_50.pkl /content/data/MSA-Dataset/aligned_50.pkl')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Verify new imports work
from training.config_phase1 import Phase1Config
from training.dataset_mosei import create_dataloaders
from training.trainer import Phase1Trainer
from training.models.mult import MulTRegressor
from training.evaluator_emotion import (
    EMOTION_NAMES, DEFAULT_THRESHOLD,
    compute_emotion_metrics,
    find_optimal_thresholds,
    compute_emotion_metrics_with_tuned_thresholds,
)
from training.losses import FocalLoss, compute_pos_weight
print('All imports OK.')

## 1. Load Checkpoint từ Phase 1 (P0 Baseline)

In [ ]:
# ============================================================
# ROUND 2A: Quick Eval — Threshold Tuning (NO RETRAINING)
# ============================================================
# Cell này load checkpoint cũ từ Phase 1 và áp dụng
# per-emotion threshold tuning để cải thiện Mean F1
# mà KHÔNG cần train lại.
# ============================================================

config = Phase1Config()
config.model_type = 'mult'
config.training.task_type = 'emotion'
config.runtime.use_drive_outputs_on_colab = USE_DRIVE_OUTPUTS
config.runtime.use_gcs = USE_GCS
config.runtime.gcs_bucket = GCS_BUCKET
config.apply_profile('colab', drive_root=DRIVE_ROOT, repo_root=REPO_PATH)

# Load P0 checkpoint từ Phase 1
P0_CHECKPOINT = config.paths.checkpoints_dir / 'best_model_mult_emotion.pt'
print(f'P0 Checkpoint: {P0_CHECKPOINT}')
print(f'Exists: {P0_CHECKPOINT.exists()}')

if not P0_CHECKPOINT.exists():
    print('WARNING: P0 checkpoint not found. Download from GCS...')
    get_ipython().system(f'gsutil cp gs://{GCS_BUCKET}/checkpoints/phase1/best_model_mult_emotion.pt {P0_CHECKPOINT}')

In [ ]:
# Load dataloaders
dataloaders = create_dataloaders(config=config, pkl_path=config.paths.mosei_pkl)

# Load P0 model — MUST use P0 architecture to match checkpoint weights
checkpoint = torch.load(P0_CHECKPOINT, map_location="cpu", weights_only=False)

# Restore P0 architecture from checkpoint so weight shapes match
saved_cfg = checkpoint.get("config", {})
p0_mult_cfg = saved_cfg.get("mult_model", {})
config.mult_model.d_model = p0_mult_cfg.get("d_model", 64)
config.mult_model.num_heads = p0_mult_cfg.get("num_heads", 4)
config.mult_model.num_cross_layers = p0_mult_cfg.get("num_cross_layers", 4)
config.mult_model.num_self_layers = p0_mult_cfg.get("num_self_layers", 2)
config.mult_model.ffn_dim = p0_mult_cfg.get("ffn_dim", 128)
config.mult_model.attn_dropout = p0_mult_cfg.get("attn_dropout", 0.2)
config.mult_model.fusion_hidden_dim = p0_mult_cfg.get("fusion_hidden_dim", 128)
config.mult_model.fusion_dropout = p0_mult_cfg.get("fusion_dropout", 0.5)
config.mult_model.output_dim = 6
config.mult_model.stochastic_depth_survival = p0_mult_cfg.get("stochastic_depth_survival", 1.0)

model = MulTRegressor(config.mult_model)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"P0 architecture: d_model={config.mult_model.d_model}, num_heads={config.mult_model.num_heads}")
print(f"Loaded P0 checkpoint: epoch {checkpoint.get("epoch", "unknown")}")
print(f"Best metric: {checkpoint.get("best_metric", "unknown")}")


In [ ]:
# ============================================================
# Inference trên validation set để tìm optimal thresholds
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

def collect_predictions(data_loader, model, device):
    """Collect all predictions and labels from a data loader."""
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            text = batch['text'].to(device)
            audio = batch['audio'].to(device)
            vision = batch['vision'].to(device)
            labels = batch['label'].to(device)
            preds = model(text=text, audio=audio, vision=vision)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    return np.concatenate(all_preds, 0), np.concatenate(all_labels, 0)

print('Collecting predictions on validation set...')
val_preds, val_labels = collect_predictions(dataloaders['valid'], model, device)
print(f'Val predictions shape: {val_preds.shape}')
print(f'Val labels shape: {val_labels.shape}')

print()
print('Collecting predictions on test set...')
test_preds, test_labels = collect_predictions(dataloaders['test'], model, device)
print(f'Test predictions shape: {test_preds.shape}')
print(f'Test labels shape: {test_labels.shape}')

In [ ]:
# ============================================================
# P0 Result: Baseline metrics với fixed threshold 0.5
# ============================================================
print('=' * 60)
print('P0 BASELINE (fixed threshold 0.5)')
print('=' * 60)

val_metrics_p0 = compute_emotion_metrics(val_labels, val_preds)
test_metrics_p0 = compute_emotion_metrics(test_labels, test_preds)

print('\n  Validation Metrics:')
for k, v in val_metrics_p0.items():
    if not k.endswith('_mae') and not k.endswith('_acc'):
        print(f'    {k}: {v:.4f}')

print('\n  Test Metrics:')
for k, v in test_metrics_p0.items():
    if not k.endswith('_mae') and not k.endswith('_acc'):
        print(f'    {k}: {v:.4f}')

In [ ]:
# ============================================================
# P0.1: Per-emotion Threshold Tuning (tìm threshold tối ưu)
# ============================================================
print('=' * 60)
print('P0.1: PER-EMOTION THRESHOLD TUNING')
print('=' * 60)
print('Grid search [0.05, 0.90] trên validation set...')

optimal_thresholds = find_optimal_thresholds(val_labels, val_preds)

print('\n  Optimal thresholds:')
for emo, thresh in optimal_thresholds.items():
    print(f'    {emo:>10}: {thresh:.2f}')

In [ ]:
# ============================================================
# Apply tuned thresholds: compare P0 vs P0.1
# ============================================================
print('=' * 60)
print('COMPARISON: P0 (fixed 0.5) vs P0.1 (tuned thresholds)')
print('=' * 60)

val_metrics_tuned = compute_emotion_metrics_with_tuned_thresholds(
    val_labels, val_preds, optimal_thresholds
)
test_metrics_tuned = compute_emotion_metrics_with_tuned_thresholds(
    test_labels, test_preds, optimal_thresholds
)

def print_comparison_table(p0_metrics, tuned_metrics):
    print(f"\n  {'Emotion':<12} {'P0 F1':>10} {'Tuned F1':>10} {'Delta':>10}")
    print(f"  {'-'*44}")
    for emo in EMOTION_NAMES:
        k = f'{emo}_f1'
        p0_f1 = p0_metrics.get(k, 0)
        t_f1 = tuned_metrics.get(k, 0)
        delta = t_f1 - p0_f1
        delta_str = f'{delta:+.4f}' if delta != 0 else '  (same)'
        print(f"  {emo:<12} {p0_f1:>10.4f} {t_f1:>10.4f} {delta_str:>10}")
    print(f"  {'-'*44}")
    print(f"  {'Mean F1':<12} {p0_metrics.get('mean_f1', 0):>10.4f} {tuned_metrics.get('mean_f1', 0):>10.4f} {tuned_metrics.get('mean_f1', 0) - p0_metrics.get('mean_f1', 0):>+10.4f}")

print('  [Validation Set]')
print_comparison_table(val_metrics_p0, val_metrics_tuned)

print()
print('  [Test Set]')
print_comparison_table(test_metrics_p0, test_metrics_tuned)

In [ ]:
# ============================================================
# Save threshold results
# ============================================================
import json

results_p01 = {
    'approach': 'P0.1: Per-emotion Threshold Tuning',
    'checkpoint': str(P0_CHECKPOINT),
    'optimal_thresholds': {k: float(v) for k, v in optimal_thresholds.items()},
    'validation': {k: float(v) for k, v in val_metrics_tuned.items()},
    'test': {k: float(v) for k, v in test_metrics_tuned.items()},
    'improvement_vs_p0': {
        'val_mean_f1_p0': float(val_metrics_p0['mean_f1']),
        'val_mean_f1_tuned': float(val_metrics_tuned['mean_f1']),
        'val_mean_f1_delta': float(val_metrics_tuned['mean_f1'] - val_metrics_p0['mean_f1']),
        'test_mean_f1_p0': float(test_metrics_p0['mean_f1']),
        'test_mean_f1_tuned': float(test_metrics_tuned['mean_f1']),
        'test_mean_f1_delta': float(test_metrics_tuned['mean_f1'] - test_metrics_p0['mean_f1']),
    }
}

output_path = config.paths.outputs_dir / 'round2a_threshold_tuning.json'
config.paths.outputs_dir.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(results_p01, indent=2))
print(f'Results saved to: {output_path}')

if USE_GCS:
    get_ipython().system(f'gsutil cp {output_path} gs://{GCS_BUCKET}/outputs/phase1/round2a_threshold_tuning.json')
    print('Uploaded to GCS.')

---

## 2. Round 2B: Train MulT P1 với Focal Loss

In [ ]:
# ============================================================
# ROUND 2B: Train MulT P1 với all P1 improvements
# ============================================================
# MulT P1 Config:
#   - d_model: 64 → 128 (4x capacity, reduce projection bottleneck)
#   - num_heads: 4 → 8 (better attention)
#   - fusion_hidden_dim: 128 → 256
#   - attn_dropout: 0.2 → 0.1
#   - fusion_dropout: 0.5 → 0.3
#   - stochastic depth: survival=0.8 (LayerDrop)
#
# Loss P1:
#   - BCEWithLogitsLoss + pos_weight (auto-computed from training labels)
#   - HOẶC FocalLoss với gamma=2, alpha=0.25
#
# ============================================================

# Choose loss type: 'bce' (with pos_weight) or 'focal'
ROUND2B_LOSS_TYPE = 'focal'  # RECOMMENDED: Focal Loss handles imbalance automatically

config_p1 = Phase1Config()
config_p1.model_type = 'mult'
config_p1.training.task_type = 'emotion'
config_p1.training.loss_type = ROUND2B_LOSS_TYPE

config_p1.runtime.use_drive_outputs_on_colab = USE_DRIVE_OUTPUTS
config_p1.runtime.use_gcs = USE_GCS
config_p1.runtime.gcs_bucket = GCS_BUCKET
config_p1.wandb.enable = WANDB_ENABLE
config_p1.wandb.project = WANDB_PROJECT
config_p1.apply_profile('colab', drive_root=DRIVE_ROOT, repo_root=REPO_PATH)

# === MulT P1 Architecture (d_model=128, num_heads=8) ===
config_p1.mult_model.d_model = 128          # P0: 64 → 128
config_p1.mult_model.num_heads = 8           # P0: 4 → 8
config_p1.mult_model.num_cross_layers = 4
config_p1.mult_model.num_self_layers = 2
config_p1.mult_model.ffn_dim = 128
config_p1.mult_model.attn_dropout = 0.1      # P0: 0.2 → 0.1
config_p1.mult_model.fusion_hidden_dim = 256  # P0: 128 → 256
config_p1.mult_model.fusion_dropout = 0.3   # P0: 0.5 → 0.3
config_p1.mult_model.stochastic_depth_survival = 0.8  # P1: LayerDrop
config_p1.mult_model.output_dim = 6           # 6 emotions

# === Training Config ===
config_p1.training.batch_size = 32
config_p1.training.num_workers = 2
config_p1.training.num_epochs = 25           # P0: 50 → 25 (P1 converges faster)
config_p1.training.patience = 7              # P0: 10 → 7
config_p1.training.scheduler_patience = 4
config_p1.training.learning_rate = 1e-4
config_p1.training.weight_decay = 3e-3      # P0: 5e-3 → 3e-3 (fewer params now OK)
config_p1.training.scheduler_type = 'cosine_warmup'
config_p1.training.warmup_epochs = 3
config_p1.training.min_lr = 1e-7
config_p1.training.max_grad_norm = 0.5
config_p1.training.use_amp = True
config_p1.training.resume_from_checkpoint = False  # Fresh training
config_p1.training.checkpoint_name = 'best_model_mult_emotion_p1_focal.pt'
config_p1.training.last_checkpoint_name = 'last_model_mult_emotion_p1_focal.pt'

# === Focal Loss Config ===
config_p1.training.focal_alpha = 0.25
config_p1.training.focal_gamma = 2.0
config_p1.training.pos_weight_max = 50.0

# Sync configs
config_p1.mult_model.stochastic_depth_survival = config_p1.training.stochastic_depth_survival

config_p1.setup()

print('=== MulT P1 Config ===')
print(f'Loss type: {config_p1.training.loss_type}')
print(f'd_model: {config_p1.mult_model.d_model}')
print(f'num_heads: {config_p1.mult_model.num_heads}')
print(f'fusion_hidden_dim: {config_p1.mult_model.fusion_hidden_dim}')
print(f'attn_dropout: {config_p1.mult_model.attn_dropout}')
print(f'stochastic_depth_survival: {config_p1.mult_model.stochastic_depth_survival}')
print(f'focal_alpha: {config_p1.training.focal_alpha}')
print(f'focal_gamma: {config_p1.training.focal_gamma}')
print(f'epochs: {config_p1.training.num_epochs}')
print(f'patience: {config_p1.training.patience}')

In [ ]:
# Smoke test: MulT P1 với output_dim=6
print('=== Smoke Test: MulTRegressor P1 (output_dim=6) ===')
model_p1 = MulTRegressor(config_p1.mult_model)

total_params = sum(p.numel() for p in model_p1.parameters())
trainable_params = sum(p.numel() for p in model_p1.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Forward/backward pass
B, T = 4, 50
text = torch.randn(B, T, 768)
audio = torch.randn(B, T, 74)
vision = torch.randn(B, T, 35)

output = model_p1(text=text, audio=audio, vision=vision)
assert output.shape == (B, 6), f'Expected (B, 6), got {output.shape}'
print(f'Forward pass OK. Output shape: {output.shape}')

# Test Focal Loss
from training.losses import FocalLoss
labels = torch.zeros(B, 6)
labels[:, 0] = 1  # Happy positive
labels[:, 3] = 1  # Surprise positive

pos_weight = torch.tensor([1.9, 6.0, 5.9, 28.0, 7.4, 43.8])
focal_loss = FocalLoss(alpha=0.25, gamma=2.0, pos_weight=pos_weight)
loss = focal_loss(output, labels)
loss.backward()
print(f'Focal Loss OK. Loss: {loss.item():.4f}')
print('All smoke tests passed!')

In [ ]:
# Load dataloaders for P1
dataloaders_p1 = create_dataloaders(config=config_p1, pkl_path=config_p1.paths.mosei_pkl)

print('Dataset sizes:')
for split, loader in dataloaders_p1.items():
    print(f'  {split}: {len(loader.dataset)} samples, {len(loader)} batches')

# Show class distribution
EMOTIONS = ['happy', 'sad', 'angry', 'surprise', 'disgust', 'fear']
sample_batch = next(iter(dataloaders_p1['train']))
label_batch = sample_batch['label']
print()
print('Class distribution in first training batch:')
for i, name in enumerate(EMOTIONS):
    col = label_batch[:, i].numpy()
    n_present = (col >= 0.5).sum()
    print(f'  {name:>10}: {n_present:>2}/{len(col)} present')

In [ ]:
# ============================================================
# TRAIN MulT P1 với Focal Loss
# ============================================================

if WANDB_ENABLE:
    import wandb
    wandb.login()

model_p1 = MulTRegressor(config_p1.mult_model)
trainer_p1 = Phase1Trainer(model=model_p1, config=config_p1)

print('Starting Round 2B training...')
print(f'Loss: {config_p1.training.loss_type}')
print(f'Model: MulT P1 (d_model=128, num_heads=8)')
print(f'Epochs: {config_p1.training.num_epochs}')
print(f'Patience: {config_p1.training.patience}')
print()

summary_p1 = trainer_p1.fit(dataloaders_p1['train'], dataloaders_p1['valid'])
test_metrics_p1 = trainer_p1.evaluate_and_save(
    dataloaders_p1['test'], split='test', epoch=summary_p1['best_epoch']
)

print()
print('=' * 60)
print('ROUND 2B: MulT P1 + Focal Loss — Results')
print('=' * 60)
print(f'Best epoch: {summary_p1["best_epoch"]}')
print(f'Best metric: {summary_p1["best_metric"]:.4f}')
print()
for k, v in test_metrics_p1.items():
    if k not in ('split', 'epoch'):
        print(f'  {k}: {v:.4f}')

## 3. Apply Threshold Tuning trên P1 Model

In [ ]:
# ============================================================
# Apply P0.1 threshold tuning lên P1 model
# ============================================================

print('Collecting predictions from P1 model...')
val_preds_p1, val_labels_p1 = collect_predictions(dataloaders_p1['valid'], model_p1, device)
test_preds_p1, test_labels_p1 = collect_predictions(dataloaders_p1['test'], model_p1, device)

# P1 metrics with fixed threshold
val_metrics_p1_fixed = compute_emotion_metrics(val_labels_p1, val_preds_p1)
test_metrics_p1_fixed = compute_emotion_metrics(test_labels_p1, test_preds_p1)

# P1.1: Threshold tuning
optimal_thresholds_p1 = find_optimal_thresholds(val_labels_p1, val_preds_p1)

val_metrics_p1_tuned = compute_emotion_metrics_with_tuned_thresholds(
    val_labels_p1, val_preds_p1, optimal_thresholds_p1
)
test_metrics_p1_tuned = compute_emotion_metrics_with_tuned_thresholds(
    test_labels_p1, test_preds_p1, optimal_thresholds_p1
)

print()
print('=' * 70)
print('FULL COMPARISON: P0 (baseline) vs P1+Focal vs P1+Thresholds')
print('=' * 70)

def print_full_comparison_table(p0, p1_fixed, p1_tuned):
    print(f"\n  {'Metric':<15} {'P0 (baseline)':>14} {'P1+Focal':>14} {'P1+Tuned':>14} {'P1 vs P0':>10}")
    print(f"  {'-'*67}")
    for emo in EMOTIONS:
        k = f'{emo}_f1'
        p0_v = p0.get(k, 0)
        p1f_v = p1_fixed.get(k, 0)
        p1t_v = p1_tuned.get(k, 0)
        delta = p1t_v - p0_v
        print(f"  {emo:<15} {p0_v:>14.4f} {p1f_v:>14.4f} {p1t_v:>14.4f} {delta:>+10.4f}")
    print(f"  {'-'*67}")
    for key in ['mean_f1', 'mean_acc']:
        p0_v = p0.get(key, 0)
        p1f_v = p1_fixed.get(key, 0)
        p1t_v = p1_tuned.get(key, 0)
        delta = p1t_v - p0_v
        print(f"  {key:<15} {p0_v:>14.4f} {p1f_v:>14.4f} {p1t_v:>14.4f} {delta:>+10.4f}")

print('  [Validation Set]')
print_full_comparison_table(val_metrics_p0, val_metrics_p1_fixed, val_metrics_p1_tuned)

print()
print('  [Test Set]')
print_full_comparison_table(test_metrics_p0, test_metrics_p1_fixed, test_metrics_p1_tuned)

## 4. Visualization

In [ ]:
# ============================================================
# Plot: Per-emotion F1 comparison
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test set F1 comparison
x = np.arange(len(EMOTIONS))
width = 0.25

p0_f1 = [test_metrics_p0.get(f'{e}_f1', 0) for e in EMOTIONS]
p1_fixed_f1 = [test_metrics_p1_fixed.get(f'{e}_f1', 0) for e in EMOTIONS]
p1_tuned_f1 = [test_metrics_p1_tuned.get(f'{e}_f1', 0) for e in EMOTIONS]

axes[0].bar(x - width, p0_f1, width, label='P0 (baseline)', color='tab:blue', alpha=0.7)
axes[0].bar(x, p1_fixed_f1, width, label='P1 + Focal (0.5)', color='tab:orange', alpha=0.7)
axes[0].bar(x + width, p1_tuned_f1, width, label='P1 + Focal + Tuned', color='tab:green', alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(EMOTIONS)
axes[0].set_ylabel('F1 Score')
axes[0].set_title('Test Set: Per-Emotion F1')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Mean F1 summary
labels = ['P0\n(baseline)', 'P1 + Focal\n(thresh=0.5)', 'P1 + Focal\n(tuned)']
p0_mean = test_metrics_p0['mean_f1']
p1f_mean = test_metrics_p1_fixed['mean_f1']
p1t_mean = test_metrics_p1_tuned['mean_f1']

bars = axes[1].bar(labels, [p0_mean, p1f_mean, p1t_mean], color=['tab:blue', 'tab:orange', 'tab:green'], alpha=0.7)
axes[1].set_ylabel('Mean F1')
axes[1].set_title('Test Set: Mean F1 Comparison')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, [p0_mean, p1f_mean, p1t_mean]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig(config_p1.paths.outputs_dir / 'round2_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to: {config_p1.paths.outputs_dir / "round2_results.png"}')

In [ ]:
# ============================================================
# Plot: Training history
# ============================================================
history_path = config_p1.paths.logs_dir / 'history.csv'
if history_path.exists():
    history_df = pd.read_csv(history_path)
    valid_df = history_df[history_df['split'] == 'valid'].copy()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    axes[0].plot(valid_df['epoch'], valid_df['loss'], label='valid_loss', marker='o', markersize=3)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Mean F1 & Accuracy
    axes[1].plot(valid_df['epoch'], valid_df['mean_f1'], label='mean_f1', marker='o', markersize=3, color='tab:orange')
    axes[1].plot(valid_df['epoch'], valid_df['mean_acc'], label='mean_acc', marker='s', markersize=3, color='tab:green')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Metric Value')
    axes[1].set_title('Validation Metrics')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print('History file not found.')

## 5. Final Summary

In [ ]:
# ============================================================
# Save final Round 2 results
# ============================================================
import json

final_results = {
    'round': '2B',
    'date': '2026-06-09',
    'config': {
        'loss_type': config_p1.training.loss_type,
        'd_model': config_p1.mult_model.d_model,
        'num_heads': config_p1.mult_model.num_heads,
        'fusion_hidden_dim': config_p1.mult_model.fusion_hidden_dim,
        'attn_dropout': config_p1.mult_model.attn_dropout,
        'fusion_dropout': config_p1.mult_model.fusion_dropout,
        'stochastic_depth_survival': config_p1.mult_model.stochastic_depth_survival,
        'focal_alpha': config_p1.training.focal_alpha,
        'focal_gamma': config_p1.training.focal_gamma,
        'epochs': config_p1.training.num_epochs,
        'batch_size': config_p1.training.batch_size,
        'learning_rate': config_p1.training.learning_rate,
    },
    'training_summary': summary_p1,
    'test_metrics_p0': {k: float(v) for k, v in test_metrics_p0.items()},
    'test_metrics_p1_fixed': {k: float(v) for k, v in test_metrics_p1_fixed.items()},
    'test_metrics_p1_tuned': {k: float(v) for k, v in test_metrics_p1_tuned.items()},
    'optimal_thresholds_p1': {k: float(v) for k, v in optimal_thresholds_p1.items()},
    'improvement_vs_p0': {
        'mean_f1_delta': float(test_metrics_p1_tuned['mean_f1'] - test_metrics_p0['mean_f1']),
        'mean_f1_pct_improvement': float((test_metrics_p1_tuned['mean_f1'] - test_metrics_p0['mean_f1']) / max(test_metrics_p0['mean_f1'], 1e-6) * 100),
    }
}

output_path = config_p1.paths.outputs_dir / 'round2b_final_results.json'
output_path.write_text(json.dumps(final_results, indent=2))
print(f'Final results saved to: {output_path}')

if USE_GCS:
    get_ipython().system(f'gsutil cp {output_path} gs://{GCS_BUCKET}/outputs/phase1/round2b_final_results.json')
    print('Uploaded to GCS.')

print()
print('=' * 60)
print('FINAL SUMMARY')
print('=' * 60)
print(f'P0 Baseline Mean F1:        {test_metrics_p0["mean_f1"]:.4f}')
print(f'P1 + Focal (thresh=0.5):   {test_metrics_p1_fixed["mean_f1"]:.4f}')
print(f'P1 + Focal + Tuned:        {test_metrics_p1_tuned["mean_f1"]:.4f}')
print()
print(f'Total improvement:          +{test_metrics_p1_tuned["mean_f1"] - test_metrics_p0["mean_f1"]:.4f}')
print(f'Pct improvement:             +{(test_metrics_p1_tuned["mean_f1"] - test_metrics_p0["mean_f1"]) / max(test_metrics_p0["mean_f1"], 1e-6) * 100:.1f}%')
print()
print('Per-emotion F1 (Test Set):')
for emo in EMOTIONS:
    p0_v = test_metrics_p0.get(f'{emo}_f1', 0)
    p1_v = test_metrics_p1_tuned.get(f'{emo}_f1', 0)
    delta = p1_v - p0_v
    print(f'  {emo:>10}: P0={p0_v:.4f}  P1={p1_v:.4f}  ({delta:+.4f})')